In [8]:
using Pkg
Pkg.activate(".")
using Distributed
using CSV, DataFrames, BSON, Random

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl - Cleaned`


In [31]:
rmprocs(workers())
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 4        # ← set as desired
addprocs(num_workers)

┌ Warning: rmprocs: process 1 not removed
└ @ Distributed C:\Users\mikul\AppData\Local\Programs\Julia-1.11.4\share\julia\stdlib\v1.11\Distributed\src\cluster.jl:1049


4-element Vector{Int64}:
 18
 19
 20
 21

In [32]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "data/median_RV.csv"
    data_column                  = "x1"
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 300
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "TVAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "Epanechnikov"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "one-sided"
    kernel_type_tvEWD            = "Epanechnikov"
    kernel_type_tvHAR            = "Epanechnikov"
    kernel_type_tvAR             = "Epanechnikov"

    alpha_level                  = 0.05
end

In [33]:
const SED_PATH = abspath("src/SED_Thresholds/SEDThresholds.jl")

"c:\\Users\\mikul\\Desktop\\Persistence of Shocks\\tvPersistence.jl - Cleaned\\src\\SED_Thresholds\\SEDThresholds.jl"

In [34]:
@everywhere include($SED_PATH)        # <— absolute path shipped to workers
@everywhere using .SEDThresholds

In [35]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

In [36]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 21:	[ Info: Performing boostrap simulation number 4
      From worker 18:	[ Info: Performing boostrap simulation number 1
      From worker 19:	[ Info: Performing boostrap simulation number 2
      From worker 20:	[ Info: Performing boostrap simulation number 3
      From worker 19:	[ Info: Bootstrap 2 generated.
      From worker 18:	[ Info: Bootstrap 1 generated.
      From worker 21:	[ Info: Bootstrap 4 generated.
      From worker 20:	[ Info: Bootstrap 3 generated.


Task (done) @0x0000016d3354ba30

In [37]:
sed_vals

4-element Vector{Vector{Float64}}:
 [NaN, 3.6966230186638066e-5, -2.039758423357489e-5, 5.496517896926131e-5, 6.809276822725824e-5, 5.9492580088258704e-5, 2.8672126357173987e-5, -0.00012977202537897768, -0.00012056925240746437, -6.042743311141807e-5  …  8.514223283552529e-7, 1.4277070790311207e-6, 1.5332521245876738e-6, 2.0605163658560157e-6, 2.104406949154759e-6, 1.950277889104597e-6, 1.7008605313489244e-6, 1.6549549883109052e-6, 1.7069354120490888e-6, 1.840409586003754e-6]
 [NaN, -2.5721320170055378e-5, -0.00022330114723503994, 6.290996405211105e-5, -4.92770517999534e-5, -5.4339981426853044e-5, -9.273974063127715e-5, -7.45411384084292e-5, -5.8311354666838634e-5, 2.7465783615449393e-6  …  -1.3288266604058991e-5, -8.784301177966168e-6, -5.681022242410769e-6, -4.806519890959928e-6, -1.7320539240132476e-6, -6.4200257375587275e-6, -3.5225927627412467e-6, -1.0923066140641443e-6, 1.057958581768216e-6, 3.5932611553191084e-6]
 [NaN, -0.00012631550597857588, -9.388488425140632e-5, -0.000147399

In [38]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

4

In [39]:
thr = SEDThresholds.compute_global_threshold(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: 5.1792205211023086e-6


In [ ]:
# Save the SED values into BSON file
BSON.@save "sed_thresholds.bson" sed_vals thr